# Deep Learning on EuroSAT

Course assignment — see `README.md` and `assignment.pdf`.

In [ ]:
!pip install torchvision matplotlib seaborn
!pip install torch torchvision torchmetrics scikit-learn matplotlib seaborn
!pip install transformers
!pip install -q peft transformers accelerate
from transformers import CLIPModel, CLIPProcessor
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
from torchvision import transforms
from torchvision.datasets import EuroSAT
from torch.utils.data import DataLoader, Subset
from google.colab import drive
from peft import LoraConfig, get_peft_model
import os
from torch.utils.tensorboard import SummaryWriter
from torchmetrics.classification import (
    MulticlassAccuracy,
    MulticlassPrecision,
    MulticlassRecall,
    MulticlassF1Score
)

import numpy as np
import random
import time
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
device = "cuda" if torch.cuda.is_available() else "cpu"
class_names = [
    'AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial',
    'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake'
]
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights
weights = ResNet18_Weights.IMAGENET1K_V1
# Frozen
model_resnet_frozen = resnet18(weights=weights)
model_resnet_frozen.fc = nn.Linear(model_resnet_frozen.fc.in_features, 10)
for p in model_resnet_frozen.parameters():
    p.requires_grad = False
for p in model_resnet_frozen.fc.parameters():
    p.requires_grad = True
# Full
model_resnet_full = resnet18(weights=weights)
model_resnet_full.fc = nn.Linear(model_resnet_full.fc.in_features, 10)

from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
weights = EfficientNet_B0_Weights.IMAGENET1K_V1
# Frozen
model_eff_frozen = efficientnet_b0(weights=weights)
model_eff_frozen.classifier[1] = nn.Linear(model_eff_frozen.classifier[1].in_features, 10)
for p in model_eff_frozen.features.parameters():
    p.requires_grad = False
for p in model_eff_frozen.classifier.parameters():
    p.requires_grad = True
# Full
model_eff_full = efficientnet_b0(weights=weights)
model_eff_full.classifier[1] = nn.Linear(model_eff_full.classifier[1].in_features, 10)

from torchvision.models import vit_b_16, ViT_B_16_Weights
weights = ViT_B_16_Weights.IMAGENET1K_V1
# Frozen
model_vit_frozen = vit_b_16(weights=weights)
model_vit_frozen.heads.head = nn.Linear(model_vit_frozen.heads.head.in_features, 10)
for p in model_vit_frozen.encoder.parameters():
    p.requires_grad = False
for p in model_vit_frozen.heads.parameters():
    p.requires_grad = True
# Full
model_vit_full = vit_b_16(weights=weights)
model_vit_full.heads.head = nn.Linear(model_vit_full.heads.head.in_features, 10)

In [ ]:
DATA_DIR = "./data"

base_transform = transforms.ToTensor()

dataset = EuroSAT(
    root=DATA_DIR,
    download=True,
    transform=base_transform
)
num_images = len(dataset)
classes = dataset.classes
num_classes = len(classes)

sample_image, sample_label = dataset[0]

print(f"Συνολικός αριθμός εικόνων: {num_images}")
print(f"Αριθμός κλάσεων: {num_classes}")
print("Κλάσεις:", classes)
print(f"Διαστάσεις εικόνας: {sample_image.shape}")

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(64, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.ToTensor()
])

eval_transform = transforms.ToTensor()

indices_per_class = defaultdict(list)

for idx, (_, label) in enumerate(dataset):
    indices_per_class[label].append(idx)

# Έλεγχος ισορροπίας
for label, idxs in indices_per_class.items():
    print(f"{classes[label]}: {len(idxs)} εικόνες")


In [ ]:
train_full_indices = []
val_indices = []
test_indices = []

for label, idxs in indices_per_class.items():
    idxs = np.array(idxs)
    np.random.shuffle(idxs)

    n = len(idxs)
    n_train = int(0.70 * n)
    n_val = int(0.15 * n)

    train_full_indices.extend(idxs[:n_train])
    val_indices.extend(idxs[n_train:n_train + n_val])
    test_indices.extend(idxs[n_train + n_val:])
train_5shot_indices = []
train_10shot_indices = []

train_full_per_class = defaultdict(list)

for idx in train_full_indices:
    _, label = dataset[idx]
    train_full_per_class[label].append(idx)

for label, idxs in train_full_per_class.items():
    np.random.shuffle(idxs)
    train_5shot_indices.extend(idxs[:5])
    train_10shot_indices.extend(idxs[:10])
def make_subset(indices, transform):
    ds = EuroSAT(root=DATA_DIR, download=False, transform=transform)
    return Subset(ds, indices)

train_full_ds = make_subset(train_full_indices, train_transform)
train_10shot_ds = make_subset(train_10shot_indices, train_transform)
train_5shot_ds = make_subset(train_5shot_indices, train_transform)

val_ds = make_subset(val_indices, eval_transform)
test_ds = make_subset(test_indices, eval_transform)
def print_split_info(name, indices):
    counts = defaultdict(int)
    for idx in indices:
        _, label = dataset[idx]
        counts[label] += 1

    print(f"\n{name}")
    print(f"Σύνολο: {len(indices)}")
    for label, count in counts.items():
        print(f"  {classes[label]}: {count}")

print_split_info("Train Full", train_full_indices)
print_split_info("Train 10-shot", train_10shot_indices)
print_split_info("Train 5-shot", train_5shot_indices)
print_split_info("Validation", val_indices)
print_split_info("Test", test_indices)



In [ ]:
def show_samples_per_class(dataset, classes, n=5):
    fig, axes = plt.subplots(len(classes), n, figsize=(n*2, len(classes)*2))

    class_count = defaultdict(int)

    for img, label in dataset:
        if class_count[label] < n:
            ax = axes[label][class_count[label]]
            ax.imshow(img.permute(1, 2, 0))
            ax.axis("off")
            if class_count[label] == 0:
                ax.set_title(classes[label])
            class_count[label] += 1

        if all(class_count[c] >= n for c in range(len(classes))):
            break

    plt.tight_layout()
    plt.show()

show_samples_per_class(dataset, classes)

class_sizes = [len(indices_per_class[i]) for i in range(num_classes)]

plt.figure(figsize=(10,4))
sns.barplot(x=classes, y=class_sizes)
plt.xticks(rotation=45)
plt.title("Κατανομή εικόνων ανά κλάση")
plt.show()

def show_augmentations(dataset, transform, idx=0, n=5):
    img, label = dataset[idx]

    fig, axes = plt.subplots(1, n, figsize=(15,3))
    for i in range(n):
        aug_img = transform(transforms.ToPILImage()(img))
        axes[i].imshow(aug_img.permute(1,2,0))
        axes[i].axis("off")

    plt.suptitle(f"Augmentations – κλάση: {classes[label]}")
    plt.show()

show_augmentations(dataset, train_transform)


In [ ]:
class CNN_model(nn.Module):
    def __init__(self):
        super(CNN_model, self).__init__()
        self.branch1 = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=5, padding=2),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.branch2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=5, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.branch3 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.branch4 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )


        self.fc1 = nn.Linear(128 * 4 * 4, 1024)
        self.dropout1 = nn.Dropout(0.3)
        self.fc2 = nn.Linear(1024, 256)
        self.dropout2 = nn.Dropout(0.3)
        self.fc3 = nn.Linear(256, 32)
        self.dropout3 = nn.Dropout(0.3)
        self.fc4 = nn.Linear(32, 10)

    def forward(self, x):
        x = self.branch1(x)
        x = self.branch2(x)
        x = self.branch3(x)
        x = self.branch4(x)

        x = torch.flatten(x, 1)
        x = self.dropout1(F.relu(self.fc1(x)))
        x = self.dropout2(F.relu(self.fc2(x)))
        x = self.dropout3(F.relu(self.fc3(x)))
        x = self.fc4(x)
        return x

In [ ]:
def train(dataloader,val_data, model, loss_fn, optimizer,lerning_rate, epochs, print_epoch, weight_d=0.0 ):
    start_time = time.time()
    best_model = model.state_dict()
    best_epoch = 0
    best_acc = 0.0
    criterion = loss_fn
    optimizer = optim.__dict__[optimizer](model.parameters(), lr=lerning_rate ,weight_decay=weight_d)
    #print(f"Optimizer: {optimizer}")
    # print(f"Learning rate: {lerning_rate}")
    # print(f"Weight decay: {weight_d}")
    train_losses = []
    train_accuracies = []
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct=0
        total = 0
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            # Υπολογισμός accuracy
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        epoch_loss = running_loss / len(dataloader)
        epoch_acc = correct / total
        train_losses.append(epoch_loss)
        train_accuracies.append(epoch_acc)
        f1_score = eval_f1_score(val_data, model,loss_fn)
        if f1_score > best_acc:
          best_acc = f1_score
          best_epoch = epoch
          best_model = model.state_dict()
          model.load_state_dict(best_model)
        if print_epoch:
          print(f"Epoch [{epoch+1}/{epochs}] - train Loss: {epoch_loss:.4f} - train Accuracy: {epoch_acc:.4f} - validation F1_score: {f1_score:.4f}")

    end_time = time.time()
    duration = end_time - start_time
    print(f"Συνολικός χρόνος εκπαίδευσης: {duration:.2f} δευτερόλεπτα")
    print(f"Best F1_score: {best_acc:.4f} - best epoch: [{best_epoch+1}/{epochs}]")
    model.load_state_dict(best_model)
    return model


In [ ]:
def eval_function(dataloader, model):
    model.eval()
    # Ορισμός μετρικών
    accuracy = MulticlassAccuracy(num_classes=10).to(device)
    precision = MulticlassPrecision(num_classes=10, average='macro').to(device)
    recall = MulticlassRecall(num_classes=10, average='macro').to(device)
    all_preds = []
    all_labels = []
    with torch.no_grad():
      for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        accuracy.update(preds, labels)
        precision.update(preds, labels)
        recall.update(preds, labels)

    acc_value = accuracy.compute()
    prec_value = precision.compute()
    recall_value = recall.compute()
    f1_score=eval_f1_score(dataloader, model, nn.CrossEntropyLoss())

    print("=" * 80)
    print(f"{'Accuracy':<20}{'Precision (macro)':<20}{'Recall (macro)':<20}{'F1 Score':<20}")
    print(f"{acc_value:<20.4f}{prec_value:<20.4f}{recall_value:<20.4f}{f1_score:<20.4f}")
    print("=" * 80)

    cm=confusion_matrix(all_labels, all_preds,normalize='true')
    class_names= dataset.classes
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(cmap= "Blues", xticks_rotation=45)
    plt.title("Confusion Matrix")
    plt.grid(False)
    plt.show()


In [ ]:
def eval_f1_score(dataloader, model, loss_fn):
    model.eval()
    f1_metric = MulticlassF1Score(num_classes=10, average='macro').to(device)
    total_loss = 0.0
    total_batches = 0

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            preds = torch.argmax(outputs, dim=1)

            loss = loss_fn(outputs, labels)
            total_loss += loss.item()
            total_batches += 1

            f1_metric.update(preds, labels)

    f1_value = f1_metric.compute()
    return f1_value.item()

In [ ]:
train_loader = DataLoader(
    train_full_ds,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

train_loader_10 = DataLoader(
    train_10shot_ds,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

train_loader_5 = DataLoader(
    train_5shot_ds,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
model_cnn = CNN_model().to(device)
model=train(train_loader,val_loader, model_cnn, nn.CrossEntropyLoss(), 'Adam', 0.001, 10,True, 0.0001)
eval_function(test_loader,model)

**Task 1.2 **

In [ ]:
model_resnet_frozen=model_resnet_frozen.to(device)
model=train(train_loader,val_loader, model_resnet_frozen, nn.CrossEntropyLoss(), 'Adam', 0.001, 10,True, 0)
eval_function(test_loader,model)

In [ ]:
model_resnet_full=model_resnet_full.to(device)
model=train(train_loader,val_loader, model_resnet_full, nn.CrossEntropyLoss(), 'Adam', 0.001, 10,True, 0)
eval_function(test_loader,model)

In [ ]:
model_eff_full=model_eff_full.to(device)
model=train(train_loader,val_loader, model_eff_full, nn.CrossEntropyLoss(), 'Adam', 0.001, 10,True, 0)
eval_function(test_loader,model)

In [ ]:
model_eff_frozen=model_eff_frozen.to(device)
model=train(train_loader,val_loader, model_eff_frozen, nn.CrossEntropyLoss(), 'Adam', 0.001, 10,True, 0)
eval_function(test_loader,model)

In [ ]:
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

transform_vit = transforms.Compose([
    transforms.Resize((224, 224)),  # εδώ γίνεται το resize "on the fly"
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_full_ds.dataset.transform = transform_vit
val_ds.dataset.transform = transform_vit
test_ds.dataset.transform = transform_vit
train_loader_vit = DataLoader(train_full_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader_vit   = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader_vit  = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)


In [ ]:
model_vit_full=model_vit_full.to(device)
model=train(train_loader,val_loader, model_vit_full, nn.CrossEntropyLoss(), 'Adam', 0.001, 10,True, 0)
eval_function(test_loader,model)

In [ ]:
model_vit_frozen=model_vit_frozen.to(device)
model=train(train_loader,val_loader, model_vit_frozen, nn.CrossEntropyLoss(), 'Adam', 0.001, 10,True, 0)
eval_function(test_loader,model)

**task 3**

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_eff_full=model_eff_full.to(device)
model=train(train_loader_10,val_loader, model_eff_full, nn.CrossEntropyLoss(), 'Adam', 0.001, 10,True, 0)
eval_function(test_loader,model)

In [ ]:
model_eff_full=model_eff_full.to(device)
model=train(train_loader_5,val_loader, model_eff_full, nn.CrossEntropyLoss(), 'Adam', 0.001, 10,True, 0)
eval_function(test_loader,model)

In [ ]:
model_resnet_full=model_resnet_full.to(device)
model=train(train_loader_10,val_loader, model_resnet_full, nn.CrossEntropyLoss(), 'Adam', 0.001, 10,True, 0)
eval_function(test_loader,model)

In [ ]:
model_resnet_full=model_resnet_full.to(device)
model=train(train_loader_5,val_loader, model_resnet_full, nn.CrossEntropyLoss(), 'Adam', 0.001, 10,True, 0)
eval_function(test_loader,model)

**task 2.1**

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
from torchvision.models import resnet18, ResNet18_Weights
weights = ResNet18_Weights.IMAGENET1K_V1
resnet = resnet18(weights=weights)
encoder = nn.Sequential(*list(resnet.children())[:-1])
for param in encoder.parameters():
    param.requires_grad = False

encoder = encoder.to(device)
encoder.eval()

In [ ]:
def extract_embeddings(dataloader, encoder):
    embeddings = []
    labels = []
    with torch.no_grad():
        for inputs, labels_batch in dataloader:
          inputs= inputs.to(device)
          feats= encoder(inputs)
          feats= feats.squeeze(-1).squeeze(-1)
          embeddings.append(feats.cpu())
          labels.append(labels_batch)
    embeddings = torch.cat(embeddings, dim=0)
    labels = torch.cat(labels, dim=0)
    return embeddings, labels


In [ ]:
support_5_embeddings, support_5_labels = extract_embeddings(train_loader_5, encoder)
support_10_embeddings, support_10_labels = extract_embeddings(train_loader_10, encoder)
test_embeddings, test_labels = extract_embeddings(test_loader, encoder)

In [ ]:
def compute_prototypes(embeddings, labels, num_classes):
  prototypes= torch.zeros(num_classes, embeddings.shape[1])
  for i in range(num_classes):
    class_emb= embeddings[labels==i]
    prototypes[i]= class_emb.mean(dim=0)
  return prototypes


In [ ]:
prototypes_5= compute_prototypes(support_5_embeddings, support_5_labels, 10)
prototypes_10= compute_prototypes(support_10_embeddings, support_10_labels, 10)

In [ ]:
def classify_euclidean(test_embeddings, prototypes):
    distances = torch.cdist(test_embeddings, prototypes)
    preds= torch.argmin(distances, dim=1)
    return preds

def classify_cosine(test_embeddings, prototypes):
    test_norm = F.normalize(test_embeddings, dim=1)
    prototypes_norm= F.normalize(prototypes, dim=1)
    similarities = torch.matmul(test_norm, prototypes_norm.T)
    preds= torch.argmax(similarities, dim=1)
    return preds

def evaluate(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    f1= f1_score(y_true, y_pred, average='macro')
    return accuracy, f1

In [ ]:
preds_euc_5= classify_euclidean(test_embeddings, prototypes_5)
preds_euc_10= classify_euclidean(test_embeddings, prototypes_10)
preds_cos_5= classify_cosine(test_embeddings, prototypes_5)
preds_cos_10= classify_cosine(test_embeddings, prototypes_10)

acc_euc_5, f1_euc_5= evaluate(test_labels, preds_euc_5)
acc_cos_5, f1_cos_5= evaluate(test_labels, preds_cos_5)
acc_euc_10, f1_euc_10= evaluate(test_labels, preds_euc_10)
acc_cos_10, f1_cos_10= evaluate(test_labels, preds_cos_10)

print("5-shot:")
print(f"Euclidean → Acc: {acc_euc_5:.4f}, F1: {f1_euc_5:.4f}")
print(f"Cosine    → Acc: {acc_cos_5:.4f}, F1: {f1_cos_5:.4f}")

print("\n10-shot:")
print(f"Euclidean → Acc: {acc_euc_10:.4f}, F1: {f1_euc_10:.4f}")
print(f"Cosine    → Acc: {acc_cos_10:.4f}, F1: {f1_cos_10:.4f}")


**task 2.2**

In [ ]:
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

model.eval()

basic_prompts = [
    f"a satellite image of {c}" for c in class_names
]

domain_prompts_1 = [
    f"an aerial view of {c} terrain" for c in class_names
]

domain_prompts_2 = [
    f"a satellite photograph showing {c} land use" for c in class_names
]

domain_prompts_3 = [
    f"top-down view of {c} area" for c in class_names
]

all_prompts_sets = [basic_prompts , domain_prompts_1 , domain_prompts_2 , domain_prompts_3]

In [ ]:
def get_text_embeddings(prompts):
    inputs = processor(text=prompts, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        text_emb = model.get_text_features(**inputs)
        text_emb = text_emb / text_emb.norm(dim=1, keepdim=True)
    return text_emb


def clip_collate_fn(batch):
    images = [item[0] for item in batch]
    labels = torch.tensor([item[1] for item in batch])
    processed_inputs = processor(images=images, return_tensors="pt", padding=True)
    return processed_inputs['pixel_values'], labels

def get_image_embeddings(dataloader):
    all_emb= []
    all_labels= []
    with torch.no_grad():
        for pixel_values_batch, labels_batch in dataloader:
            pixel_values_batch = pixel_values_batch.to(device)
            img_emb = model.get_image_features(pixel_values=pixel_values_batch)
            img_emb = img_emb / img_emb.norm(dim=1, keepdim=True)
            all_emb.append(img_emb.cpu())
            all_labels.append(labels_batch)
    return torch.cat(all_emb), torch.cat(all_labels)

def clip_zero_shot_classification(image_embeddings, text_embeddings):
    image_embeddings = image_embeddings.to(device)
    text_embeddings = text_embeddings.to(device)
    logit_scale = model.logit_scale.exp()

    logits_per_image = logit_scale * (image_embeddings @ text_embeddings.T)

    preds = torch.argmax(logits_per_image, dim=1)
    return preds

In [ ]:
DATA_DIR = "./data"


clip_dataset = EuroSAT(
    root=DATA_DIR,
    download=True,
    transform=None
)

# Classes
class_names = clip_dataset.classes
num_classes = len(class_names)
print("Κλάσεις:", class_names)
print("Συνολικός αριθμός εικόνων:", len(clip_dataset))

test_ds_clip = Subset(clip_dataset, range(24000, 27000))

test_loader_clip = DataLoader(
    test_ds_clip,
    batch_size=32,
    shuffle=False,
    collate_fn=clip_collate_fn
)

In [ ]:
text_emb_basic = get_text_embeddings(basic_prompts)
text_emb_prom_1= get_text_embeddings(domain_prompts_1)
text_emb_prom_2= get_text_embeddings(domain_prompts_2)
text_emb_prom_3= get_text_embeddings(domain_prompts_3)
#text_emb_all= get_text_embeddings(all_prompts)
test_img_emb, test_labels = get_image_embeddings(test_loader_clip)

text_emb_list = []
for prompts in all_prompts_sets:
    emb = get_text_embeddings(prompts)  # [num_classes, D]
    text_emb_list.append(emb)

text_emb_stack = torch.stack(text_emb_list)
text_emb_ensemble = text_emb_stack.mean(dim=0)
text_emb_ensemble = text_emb_ensemble / text_emb_ensemble.norm(dim=1, keepdim=True)
preds = clip_zero_shot_classification(test_img_emb, text_emb_ensemble)
print(torch.bincount(preds))
acc, f1= evaluate(test_labels.cpu(), preds.cpu())
print(f"CLIP Ensemble \u2192 Acc: {acc:.4f}, F1: {f1:.4f}")




pred_basic = clip_zero_shot_classification(test_img_emb, text_emb_basic)
pred_prom_1 = clip_zero_shot_classification(test_img_emb, text_emb_prom_1)
pred_prom_2 = clip_zero_shot_classification(test_img_emb, text_emb_prom_2)
pred_prom_3 = clip_zero_shot_classification(test_img_emb, text_emb_prom_3)
#pred_all = clip_zero_shot_classification(test_img_emb, text_emb_all)


acc_basic, f1_basic= evaluate(test_labels.cpu(), pred_basic.cpu())
acc_prom_1, f1_prom_1= evaluate(test_labels.cpu(), pred_prom_1.cpu())
acc_prom_2, f1_prom_2= evaluate(test_labels.cpu(), pred_prom_2.cpu())
acc_prom_3, f1_prom_3= evaluate(test_labels.cpu(), pred_prom_3.cpu())
#acc_all, f1_all= evaluate(test_labels.cpu(), pred_all.cpu())

print(f"CLIP Basic Prompt    \u2192 Acc: {acc_basic:.4f}, F1: {f1_basic:.4f}")
print(f"CLIP Domain Prompt 1 \u2192 Acc: {acc_prom_1:.4f}, F1: {f1_prom_1:.4f}")
print(f"CLIP Domain Prompt 2 \u2192 Acc: {acc_prom_2:.4f}, F1: {f1_prom_2:.4f}")
print(f"CLIP Domain Prompt 3 \u2192 Acc: {acc_prom_3:.4f}, F1: {f1_prom_3:.4f}")
#print(f"CLIP All Prompts     \u2192 Acc: {acc_all:.4f}, F1: {f1_all:.4f}")

**task 3**

In [ ]:
class ContrastiveTransformations:
    def __init__(self, base_transforms, n_views=2):
        self.base_transforms = base_transforms
        self.n_views = n_views

    def __call__(self, x):  ##λειτουργία ως  function if obj = MyClass()    τότε το obj(x) καλει την __call__
        # Εφαρμόζει τα ίδια transforms δύο φορές σε τυχαίες παραλλαγές   και τα επιστρέφει σε λίστα
        return [self.base_transforms(x) for _ in range(self.n_views)]


contrast_transforms = transforms.Compose([
    transforms.RandomResizedCrop(size=96), #resize στο 96×96. Μαθαίνει scale invariance
    transforms.RandomHorizontalFlip(), #Κατοπτρισμός εικόνας.
    transforms.RandomApply([
        transforms.ColorJitter( #Μαθαίνει το μοντέλο να μην «κλέβει» από χρώματα.
            brightness=0.5,
            contrast=0.5,
            saturation=0.5,
            hue=0.1
        )
    ], p=0.8), # ΜΟΝΟ στο 80% των περιπτώσεων.
    transforms.RandomGrayscale(p=0.2), #Με πιθανότητα 20% η εικόνα γίνεται μαυρόασπρη.
    transforms.GaussianBlur(kernel_size=9), #Προσθέτει θόλωση όπως στο original SimCLR kerlensize=9
    transforms.ToTensor(), ##Μετατρέπει σε tensor 3×96×96.
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ) #Κεντράρει και κλιμακώνει.
])

In [ ]:
class SimCLRModel(nn.Module):
    def __init__(self, feature_dim=128):
        super().__init__()

        # Base encoder f(.) εξάγει features
        # self.encoder = resnet18(pretrained=False)
        self.encoder = resnet18(weights=None)
        self.encoder.fc = nn.Identity()
        enc_dim = 512  # ResNet18 output και παίρνουμε το 512-dimensional embedding. Αυτό θα το χρησιμοποιήσεις στο projection head:
        self.projector = nn.Sequential(
            nn.Linear(enc_dim, enc_dim),
            nn.ReLU(inplace=True),
            nn.Linear(enc_dim, feature_dim)
        )

    def forward(self, x):
        # εισοδος Β × 3 × 224 × 224 και παράγει εξοδο B × 512
        h = self.encoder(x)
        # εξοδο B × 128
        z = self.projector(h)
        z = F.normalize(z, dim=1)
        return z


In [ ]:
def nt_xent_loss(z_i, z_j, temperature=0.5):

    # Batch size
    N = z_i.size(0)

    # Concatenate: 2N × D καθετα ωστε να γινουν ενα ενιαιο batch
    z = torch.cat([z_i, z_j], dim=0)

    # Similarity matrix (εξοδος 2N × 2N)  dot.product για να φτιαξει τον
    #Similarity matrix Διαιρω με τεμπ για πιο "αιχμηρο" αποτελεσμα πχ τ=0.01
    sim = torch.matmul(z, z.T) / temperature

    # Remove diagonal (self-similarities)
    #torch.eye 1 στη διαγώνιο 0 παντού αλλού
    mask = torch.eye(2*N, dtype=torch.bool, device=z.device)
    # Αντικαθιστα self similarities (αρχική τιμή 1) με πολυ μικρό αριθμό
    sim = sim.masked_fill(mask, -1e9)

    pos = torch.cat([sim.diag(N), sim.diag(-N)], dim=0)
    numerator = torch.exp(pos)
    denominator = torch.sum(torch.exp(sim), dim=1)
    return -torch.log(numerator / denominator).mean()

In [ ]:
def train_simclr(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0
    i=0
    for batch in dataloader:
        i=i+1
        print ("batch-no=",i)

        # Το dataloader δίνει δύο augmented views για κάθε εικόνα
        imgs, _ = batch
        #Αν batch size 256
        #x_i = 256 augmented images
        #x_j = 256 άλλες augmented images
        x_i, x_j = imgs[0].to(device), imgs[1].to(device)
        #print(x_i.shape)
        #print(x_j.shape)
        # Περναν από το μοντέλο οι εικόνες
        #Προβλέψεις του μοντέλου για τα δύο views
        z_i = model(x_i)
        z_j = model(x_j)
        loss = nt_xent_loss(z_i, z_j)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    # Μέσο loss ανά batch στο epoch
    return total_loss / len(dataloader)

In [ ]:
drive.mount('/content/drive')
log_dir = "/content/drive/MyDrive/tensorboard_logs/simclr_experiment_1"
os.makedirs(log_dir, exist_ok=True)
# writer = SummaryWriter(log_dir="runs/simclr_experiment_1")
writer = SummaryWriter(log_dir=log_dir)

In [ ]:
model = SimCLRModel(feature_dim=256).to(device)
tranform_Sim = ContrastiveTransformations(contrast_transforms)

# Using the make_subset function defined earlier
train_full_ds_sim = make_subset(train_full_indices, tranform_Sim)
validation_ds_sim = make_subset(val_indices, tranform_Sim)
test_ds_sim = make_subset(test_indices, tranform_Sim)
train5_ds_sim = make_subset(train_5shot_indices, tranform_Sim)
train10_ds_sim = make_subset(train_10shot_indices, tranform_Sim)

train_loader_sim = DataLoader(train_full_ds_sim, batch_size=124, shuffle=True)
test_loader_sim = DataLoader(test_ds_sim, batch_size=124, shuffle=False)
# Optimizer για την εκπαίδευση του contrastive objective
optimizer = optim.Adam(model.parameters(), lr=5e-4)

# Training loop: κάθε epoch υπολογίζει το NT-Xent loss στο unlabeled set
print("Αριθμός  batch = ",len(train_full_ds_sim)) # Corrected to use the SimCLR dataset length
for epoch in range(40):
    print(epoch)
    loss = train_simclr(model, train_loader_sim, optimizer, device) # Corrected to pass train_loader_sim
    print(f"Epoch {epoch} - Loss: {loss:.4f}")

    # -------------------------------------------------------
    # 2. Log στο TensorBoard
    # -------------------------------------------------------
    writer.add_scalar("Loss/train", loss, epoch)

writer.close()

In [ ]:
SAVE_PATH = "/content/drive/MyDrive/simclr_models/simclr_model.pth"
os.makedirs("/content/drive/MyDrive/simclr_models", exist_ok=True)
#all model
torch.save(model.state_dict(), SAVE_PATH)
#only encoder
torch.save(model.encoder.state_dict(), "/content/drive/MyDrive/simclr_models/simclr_encoder.pth")


In [ ]:
import torch

# Load the state dictionary from the saved file
encoder_state_dict = torch.load("/content/drive/MyDrive/simclr_models/simclr_encoder.pth")

print("Keys in the loaded encoder state dictionary:")
for key in encoder_state_dict.keys():
    print(key)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = resnet18(weights=None)
encoder.fc = nn.Identity()
encoder.load_state_dict(
    torch.load("/content/drive/MyDrive/simclr_models/simclr_encoder.pth",
               map_location=device)
)

encoder = encoder.to(device)
encoder.eval()

embeddings = []
labels_list = []

with torch.no_grad():
    for inputs, labels in test_loader_vit:
        inputs = inputs.to(device)
        feats = encoder(inputs)
        embeddings.append(feats.cpu().numpy())
        labels_list.append(labels.numpy())

embeddings = np.concatenate(embeddings, axis=0)
labels_list = np.concatenate(labels_list, axis=0)

tsne = TSNE(n_components=2, perplexity=30, random_state=42)
emb_2d = tsne.fit_transform(embeddings)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(
    emb_2d[:, 0],
    emb_2d[:, 1],
    c=labels_list,
    cmap="tab10",
    alpha=0.7
)
plt.legend(*scatter.legend_elements(), title="Classes")
plt.title("t-SNE of SimCLR Embeddings (Test Set)")
plt.show()

In [ ]:
classifier = nn.Linear(512, 10).to(device)
encoder = resnet18(weights=None)
encoder.fc = nn.Identity()

encoder.load_state_dict(
    torch.load("/content/drive/MyDrive/simclr_models/simclr_encoder.pth",
               map_location=device)
)
encoder = encoder.to(device)
class LinearEvalModel(nn.Module):
    def __init__(self, encoder, classifier):
        super().__init__()
        self.encoder = encoder
        self.classifier = classifier

    def forward(self, x):
        x = self.encoder(x)
        return self.classifier(x)

model = LinearEvalModel(encoder, classifier).to(device)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_1= train(train_loader,val_loader, model, nn.CrossEntropyLoss(), 'Adam', 0.001, 10,True)
eval_function(test_loader,model_1)

In [ ]:
model_2= train(train_loader_10,val_loader, model, nn.CrossEntropyLoss(), 'Adam', 0.001, 10,True)
eval_function(test_loader,model_2)

In [ ]:
model_3= train(train_loader_5,val_loader, model, nn.CrossEntropyLoss(), 'Adam', 0.001, 10,True)
eval_function(test_loader,model_3)

**task 4**

In [ ]:
from peft import PeftModel

def train_vit(dataloader, val_data, model, loss_fn,
              optimizer_name, lerning_rate, epochs, print_epoch, weight_d=0.0):

    start_time = time.time()
    criterion = loss_fn
    optimizer = optim.__dict__[optimizer_name](
        model.parameters(), lr=lerning_rate, weight_decay=weight_d
    )

    best_acc = 0.0
    for epoch in range(epochs):
        torch.set_grad_enabled(True)
        model.train()

        running_loss = 0.0
        correct = 0
        total = 0

        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            logits = outputs.logits if hasattr(outputs, "logits") else outputs

            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(logits, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        epoch_loss = running_loss / len(dataloader)
        epoch_acc = correct / total

        f1_score = eval_f1_score_vit(val_data, model, loss_fn)
        if print_epoch:
            print(
                f"Epoch [{epoch+1}/{epochs}] - "
                f"train Loss: {epoch_loss:.4f} - "
                f"train Accuracy: {epoch_acc:.4f} - "
                f"validation F1_score: {f1_score:.4f}"
            )

    end_time = time.time()
    print(f"Συνολικός χρόνος εκπαίδευσης: {end_time - start_time:.2f} δευτερόλεπτα")

    return model



def eval_function_vit(dataloader, model):
    model.eval()
    # Ορισμός μετρικών
    accuracy = MulticlassAccuracy(num_classes=10).to(device)
    precision = MulticlassPrecision(num_classes=10, average='macro').to(device)
    recall = MulticlassRecall(num_classes=10, average='macro').to(device)
    all_preds = []
    all_labels = []
    with torch.no_grad():
      for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        # Extract logits if outputs is a transformers.modeling_outputs.ImageClassifierOutput
        logits = outputs.logits if hasattr(outputs, 'logits') else outputs
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        accuracy.update(preds, labels)
        precision.update(preds, labels)
        recall.update(preds, labels)

    acc_value = accuracy.compute()
    prec_value = precision.compute()
    recall_value = recall.compute()
    f1_score=eval_f1_score_vit(dataloader, model, nn.CrossEntropyLoss())

    print("=" * 80)
    print(f"{'Accuracy':<20}{'Precision (macro)':<20}{'Recall (macro)':<20}{'F1 Score':<20}")
    print(f"{acc_value:<20.4f}{prec_value:<20.4f}{recall_value:<20.4f}{f1_score:<20.4f}")
    print("=" * 80)

    cm=confusion_matrix(all_labels, all_preds,normalize='true')
    class_names= dataset.classes
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(cmap= "Blues", xticks_rotation=45)
    plt.title("Confusion Matrix")
    plt.grid(False)
    plt.show()


def eval_f1_score_vit(dataloader, model, loss_fn):
    model.eval()
    f1_metric = MulticlassF1Score(num_classes=10, average='macro').to(device)
    total_loss = 0.0
    total_batches = 0

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            # Extract logits if outputs is a transformers.modeling_outputs.ImageClassifierOutput
            logits = outputs.logits if hasattr(outputs, 'logits') else outputs
            preds = torch.argmax(logits, dim=1)

            loss = loss_fn(logits, labels)
            total_loss += loss.item()
            total_batches += 1

            f1_metric.update(preds, labels)

    f1_value = f1_metric.compute()
    return f1_value.item()

In [ ]:
!pip install -q transformers peft accelerate
import torch
import torch.nn as nn
from transformers import ViTForImageClassification, ViTImageProcessor
from peft import LoraConfig, get_peft_model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "google/vit-base-patch16-224"


vit_model = ViTForImageClassification.from_pretrained(
    model_name,
    num_labels=10,
    ignore_mismatched_sizes=True # Add this line to handle size mismatch in the classifier head
).to(device)
# Removed the manual freezing of vit_model.vit.parameters() here.
# get_peft_model will handle setting requires_grad=False for base model parameters.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
)

model_vit_lora = get_peft_model(vit_model, lora_config)
model_vit_lora = model_vit_lora.to(device)
model_vit_lora.print_trainable_parameters()

In [ ]:
model_vit_lora=train_vit(train_loader_vit,val_loader_vit, model_vit_lora, nn.CrossEntropyLoss(), 'Adam', 0.001, 7,True)
eval_function_vit(test_loader_vit,model_vit_lora)
